Final submission dosyası

Yarışmaya gönderilen son model. İki parçadan oluşuyor: LightGBM tahmini ve trafonun kendi
son 30 günlük ortalaması. İkisi log uzayında 70/30 ağırlıkla harmanlanıyor.

Buraya nasıl geldiğimizin özeti:

| Adım | Değişiklik | Kaggle |
|---|---|---|
| Başlangıç | Direct forecasting, çoklu origin | 1.13709 |
| Yaz origin'leri eklendi | Eğitim test mevsimini gördü | 1.08157 |
| yilin_gunu çıkarıldı, yaprak 63'ten 31'e | Basitleştirme | 1.06168 |
| Origin seti genişletildi | 6 origin | 1.05487 |
| Yaprak 31'den 15'e | Basitleştirme | 1.05382 |
| Tek hava sütunu (sic_7gun) | Sıcaklık | 1.05008 |
| Baz çizgisi harmanı | Bu notebook | (final) |

Denenip elenen yaklaşımlar README.md dosyasında listeli.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

RAW, EXT = '../data/raw/', '../data/external/'
train = pd.read_csv(RAW + 'train.csv')
test  = pd.read_csv(RAW + 'test.csv')
train['tarih'] = pd.to_datetime(train['tarih'])
test['tarih']  = pd.to_datetime(test['tarih'])

# Kurulu gücün 24 saatlik teorik tavanını aşan 444 satır var. Bunların yüzde 78'i sadece
# 4 trafoda toplandığı için genel bir kırpma yerine trafo bazlı medyanla düzeltiyoruz.
train['asiri_oran'] = train['tuketim'] / (train['guc'] * 24)
train['tuketim_duzeltilmis'] = train['tuketim']
for t in train.loc[train['asiri_oran'] > 2, 'tanim'].unique():
    medyan = train.loc[(train['tanim'] == t) & (train['asiri_oran'] <= 2), 'tuketim'].median()
    train.loc[(train['tanim'] == t) & (train['asiri_oran'] > 2), 'tuketim_duzeltilmis'] = medyan

print('düzeltilen satır:', (train['tuketim'] != train['tuketim_duzeltilmis']).sum())

In [ ]:
TATILLER = pd.to_datetime([
    '2025-01-01','2025-03-29','2025-03-30','2025-03-31','2025-04-01','2025-04-23','2025-05-01',
    '2025-05-19','2025-06-05','2025-06-06','2025-06-07','2025-06-08','2025-06-09','2025-08-30','2025-10-29',
    '2026-01-01','2026-03-19','2026-03-20','2026-03-21','2026-04-23','2026-05-01','2026-05-19',
    '2026-05-26','2026-05-27','2026-05-28','2026-05-29','2026-05-30','2026-08-30','2026-10-29',
]).date

GUC_BINS = [0, 100, 250, 500, 1000, 2000, 5000, np.inf]


def temel_ozellikler(df):
    df = df.copy()
    lok = df['lokasyon'].str.split('>', expand=True)
    df['il']    = lok[0]
    df['bolge'] = lok[1]
    # Manisa kayıtlarında ilçe kırılımı yok, orada bölge adı hem kategori hem hava anahtarı
    df['ilce']  = (lok[2] if lok.shape[1] > 2 else pd.Series(np.nan, index=df.index)).fillna('YOK')
    df['ay']            = df['tarih'].dt.month
    df['gun']           = df['tarih'].dt.day
    df['haftanin_gunu'] = df['tarih'].dt.dayofweek
    df['hafta_sonu']    = (df['haftanin_gunu'] >= 5).astype(int)
    df['tatil']         = df['tarih'].dt.date.isin(TATILLER).astype(int)
    df['guc_bucket']     = pd.cut(df['guc'], bins=GUC_BINS)
    df['guc_bucket_str'] = df['guc_bucket'].astype(str)
    df['yer'] = np.where(df['ilce'] != 'YOK', df['ilce'], df['bolge'])
    return df


train = temel_ozellikler(train)
test  = temel_ozellikler(test)
train['log_t'] = np.log1p(train['tuketim_duzeltilmis'])
KATEGORIK = ['il', 'bolge', 'ilce', 'guc_bucket_str']
print('train', train.shape, '| test', test.shape)

## Sıcaklık

Open Meteo arşiv servisinden 47 ilçe için günlük sıcaklık çektik. Test dönemi geçmişte
kaldığı için tahmin edilmiş değil gerçekleşmiş sıcaklıkları kullanabiliyoruz. Çekim kodu
03_deneyler.ipynb dosyasında, sonuç data/external klasöründe duruyor.

Modele tek bir sıcaklık sütunu koyuyoruz: 7 günlük hareketli ortalama. Önce 15 sütunluk
bir paket denedik (ham sıcaklıklar, derece gün, farklı pencereler) ve Kaggle skoru
kötüleşti. Üç doğrulamada tek tek ölçünce en değerli sütunun bu olduğu çıktı.

7 günlük ortalamayı günlük sıcaklığa tercih etmemizin sebebi termal atalet. Binalar bir
günde ısınıp soğumuyor, üç gündür sıcaksa klimalar sürekli çalışıyor demektir. Ham günlük
sıcaklık her doğrulamada 7 günlük ortalamanın gerisinde kaldı.

Bir nokta önemli: yilin_gunu özelliğini modelden çıkarmıştık çünkü eğitim ayları ile test
ayları arasında değer örtüşmesi yok, ağaçlar dış değer tahmini yapamıyor. Sıcaklıkta böyle bir
sorun yok, test aralığı eğitim aralığının içinde kalıyor.

In [ ]:
hava = pd.read_csv(EXT + 'hava_durumu.csv', parse_dates=['tarih'])
hava = hava.sort_values(['yer', 'tarih']).reset_index(drop=True)
hava['sic_7gun'] = hava.groupby('yer')['sic_ort'].transform(lambda s: s.rolling(7, min_periods=1).mean())

train = train.merge(hava[['yer', 'tarih', 'sic_7gun']], on=['yer', 'tarih'], how='left')
test  = test.merge(hava[['yer', 'tarih', 'sic_7gun']],  on=['yer', 'tarih'], how='left')

print('eksik sıcaklık, train:', train['sic_7gun'].isna().sum(), '| test:', test['sic_7gun'].isna().sum())
print('sic_7gun train aralığı: %.1f ile %.1f' % (train['sic_7gun'].min(), train['sic_7gun'].max()))
print('sic_7gun test aralığı : %.1f ile %.1f' % (test['sic_7gun'].min(), test['sic_7gun'].max()))

## Özellikler

Hepsi origin tarihinden önceki veriden hesaplanıyor. Sızıntı koruması tek başına buna dayalı.

Grup istatistiği il, bölge ve güç kovası bazında log ortalama. Küçük gruplarda ham ortalama
gürültülü olduğu için shrinkage uyguluyoruz, k değeri 30. Eşleşmeyen kombinasyonlar için
önce il artı güç, sonra genel ortalama yedeği var.

Trafo profili ortalama, medyan, gün sayısı, log ortalama, log medyan, log standart sapma,
sıfır oranı, son 30 ve son 90 günün log ortalaması ve sıfır oranı, kısa ile uzun vade farkı.

Hafta günü profili trafonun kendi ortalamasına göre gün bazlı oranı. Sanayi trafosu ile
konut trafosunu ayırt etmeye yarıyor.

In [ ]:
def trafo_profili(gecmis, origin):
    g = gecmis[gecmis['tarih'] < origin]
    profil = g.groupby('tanim').agg(
        trafo_ortalama   = ('tuketim_duzeltilmis', 'mean'),
        trafo_medyan     = ('tuketim_duzeltilmis', 'median'),
        trafo_gun_sayisi = ('tuketim_duzeltilmis', 'count'),
        log_ortalama     = ('log_t', 'mean'),
        log_medyan       = ('log_t', 'median'),
        log_std          = ('log_t', 'std'),
        sifir_orani      = ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
    ).reset_index()
    for gun in [30, 90]:
        son = g[g['tarih'] >= origin - pd.Timedelta(days=gun)]
        ozet = son.groupby('tanim').agg(**{
            f'son{gun}_log_ort': ('log_t', 'mean'),
            f'son{gun}_sifir'  : ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
        }).reset_index()
        profil = profil.merge(ozet, on='tanim', how='left')
    profil['log_trend'] = profil['son30_log_ort'] - profil['son90_log_ort']
    return profil


def trafo_gun_profili(gecmis, origin):
    g = gecmis[gecmis['tarih'] < origin][['tanim', 'haftanin_gunu', 'tuketim_duzeltilmis']].copy()
    ort = g.groupby('tanim')['tuketim_duzeltilmis'].mean().rename('_ort')
    g = g.merge(ort, on='tanim', how='left')
    g['oran'] = g['tuketim_duzeltilmis'] / g['_ort'].replace(0, np.nan)
    return g.groupby(['tanim', 'haftanin_gunu'])['oran'].mean().rename('trafo_gun_orani').reset_index()


def veri_seti_kur(hedef, origin, k=30):
    g = train[train['tarih'] < origin]
    genel = g['log_t'].mean()

    grup = g.groupby(['il', 'bolge', 'guc_bucket'], observed=True)['log_t'].agg(['sum', 'count']).reset_index()
    grup['grup_log_ort'] = (grup['sum'] + k * genel) / (grup['count'] + k)
    il_guc = g.groupby(['il', 'guc_bucket'], observed=True)['log_t'].mean().rename('ilguc_log_ort').reset_index()

    df = hedef.merge(grup[['il', 'bolge', 'guc_bucket', 'grup_log_ort']],
                     on=['il', 'bolge', 'guc_bucket'], how='left')
    df = df.merge(il_guc, on=['il', 'guc_bucket'], how='left')
    df['grup_log_ort'] = df['grup_log_ort'].fillna(df['ilguc_log_ort']).fillna(genel)
    df = df.merge(trafo_profili(train, origin), on='tanim', how='left')
    df = df.merge(trafo_gun_profili(train, origin), on=['tanim', 'haftanin_gunu'], how='left')
    return df


def origin_seti(bas, bit, origin):
    return veri_seti_kur(train[(train['tarih'] >= bas) & (train['tarih'] <= bit)], pd.Timestamp(origin))


OZELLIKLER = [
    'guc', 'ay', 'gun', 'haftanin_gunu', 'hafta_sonu', 'tatil',
    'il', 'bolge', 'ilce', 'guc_bucket_str', 'grup_log_ort',
    'trafo_ortalama', 'trafo_medyan', 'trafo_gun_sayisi',
    'log_ortalama', 'log_medyan', 'log_std', 'sifir_orani',
    'son30_log_ort', 'son90_log_ort', 'son30_sifir', 'son90_sifir',
    'log_trend', 'trafo_gun_orani', 'sic_7gun',
]
print('özellik sayısı:', len(OZELLIKLER))

## Origin seçimi

Bir origin, elimizde şu tarihe kadar veri var sonrasını tahmin et demek. Aynı hedef dönemi
farklı origin'lerle tekrar üretip yığıyoruz.

Yaz origin'leri (Nisan, Mayıs, Haziran 2025, hedef 30 Eylül'e kadar) modelin sıcak hava
davranışını görmesini sağlıyor. Bunlar eklendiğinde skor 1.13709'dan 1.08157'e düştü,
yarışmadaki en büyük tek kazanç buydu.

Kış origin'leri (Aralık 2025, Ocak ve Şubat 2026, hedef 31 Mart'a kadar) zengin profillere
sahip, 11 ile 13 aylık geçmiş içeriyorlar. Gerçek testteki duruma en yakın örnekler bunlar.

Nisan 2025 öncesindeki origin'ler denendi ve zarar verdi. O tarihlerde profiller sadece 1
veya 3 aylık. Daha fazla veri her zaman daha iyi değil, profil kalitesi veri miktarından
önemli çıktı.

In [ ]:
ORIGINLER = [
    ('2025-04-01', '2025-09-30'),
    ('2025-05-01', '2025-09-30'),
    ('2025-06-01', '2025-09-30'),
    ('2025-12-01', '2026-03-31'),
    ('2026-01-01', '2026-03-31'),
    ('2026-02-01', '2026-03-31'),
]

egitim = pd.concat([origin_seti(bas, bit, bas) for bas, bit in ORIGINLER], ignore_index=True)
egitim['hedef_log'] = np.log1p(egitim['tuketim_duzeltilmis'])
for col in KATEGORIK:
    egitim[col] = egitim[col].astype('category')

print('eğitim:', egitim.shape)
print('eğitim ayları:', sorted(egitim['ay'].unique()))
print('test ayları  :', sorted(test['ay'].unique()))

## Model

num_leaves değeri 15. Bu kurguda basitleştirme sürekli kazandırdı, Kaggle'da 63'ten 31'e ve
31'den 15'e her adım skoru iyileştirdi. 122 günlük ufukta model çabuk ezberliyor, basit
ağaçlar trafoya özel örüntüleri ezberleyemediği için uzak ufukta daha iyi genelliyor.

Üç farklı tohumla eğitip log uzayında ortalama alıyoruz. Metrik log ölçekte hesaplandığı
için ortalamayı da orada almak gerekiyor.

In [ ]:
AYAR = dict(num_leaves=15, learning_rate=0.05, n_estimators=87)
TOHUMLAR = [42, 7, 2024]

test_seti = veri_seti_kur(test, pd.Timestamp('2026-04-01'))

# LightGBM kategorileri kod numarasıyla kullanıyor. Test'in kategori sıralaması eğitimden
# farklı olursa model sessizce yanlış tahmin üretir, hata vermez.
for col in KATEGORIK:
    test_seti[col] = pd.Categorical(test_seti[col], categories=egitim[col].cat.categories)

print('kategori uyumu:', all(list(test_seti[c].cat.categories) == list(egitim[c].cat.categories)
                             for c in KATEGORIK))

log_tahminler = []
for tohum in TOHUMLAR:
    model = lgb.LGBMRegressor(random_state=tohum, verbose=-1, **AYAR)
    model.fit(egitim[OZELLIKLER], egitim['hedef_log'], categorical_feature=KATEGORIK)
    log_tahminler.append(model.predict(test_seti[OZELLIKLER]))

model_log = np.mean(log_tahminler, axis=0)
print('model tahmini hazır, log ortalama: %.3f' % model_log.mean())

## Baz çizgisi harmanı

Özellik önemine baktığımızda modelin kazancının yüzde 79'unun tek bir özellikten geldiğini
gördük, son30_log_ort. Permütasyon testinde bu özelliği karıştırmak skoru 1.74 bozuyor,
ikinci sıradaki özellik 0.06'da kalıyor. Yani model esasen trafonun son 30 gün ortalamasını
alıp üzerine küçük düzeltmeler yapan bir yapı.

Bunu fark edince şu soruyu sorduk: model, o ortalamayı doğrudan tahmin etmekten daha mı
iyi. Dört ayrı doğrulamada ölçtük ve iki tanesinde basit ortalama modeli yendi.

Çözüm ikisini birleştirmek oldu. Log uzayında 70 model, 30 baz çizgisi. Dört doğrulamada da
bu harman modelden iyi çıktı:

| Doğrulama | Model | Harman | Kazanç |
|---|---|---|---|
| Zengin profil, Ocak Mart | 0.7600 | 0.7451 | 0.0149 |
| Kış, Aralık Mart | 1.0688 | 1.0001 | 0.0687 |
| Yaz sonu, Ağustos Kasım | 1.0313 | 1.0189 | 0.0124 |
| İlkbahar yaz, Nisan Temmuz | 1.0131 | 0.9872 | 0.0259 |

Son satır önemli. O kurulum gerçek testin birebir analogu, aynı aylar ve aynı 122 günlük
ufuk. Orada ay bazında baktığımızda dört ayın dördünde de iyileşme çıktı, en büyüğü de
Temmuz'da, yani baz çizgisinden en uzak ayda.

Cold start trafolarda son30_log_ort boş oluyor, orada grup ortalamasına düşüyoruz.

In [ ]:
AGIRLIK = 0.70

baz_log = test_seti['son30_log_ort'].fillna(test_seti['grup_log_ort']).values
print('baz çizgisi dolu oran:', round(test_seti['son30_log_ort'].notna().mean(), 4))
print('boş kalan:', int(np.isnan(baz_log).sum()))

harman_log = AGIRLIK * model_log + (1 - AGIRLIK) * baz_log
tahmin = np.clip(np.expm1(harman_log), 0, None)   # negatif tahminler yarışma kuralı gereği sıfır

submission = pd.DataFrame({'id': test_seti['id'], 'tuketim': tahmin})
submission.to_csv('../submissions/sub_36_harman_w070.csv', index=False)

ornek = pd.read_csv(RAW + 'sample_submission.csv')
print()
print('satır sayısı eşit :', len(submission) == len(ornek))
print('id kümeleri aynı  :', set(submission['id']) == set(ornek['id']))
print('id benzersiz      :', submission['id'].is_unique)
print('NaN yok           :', submission['tuketim'].isna().sum() == 0)
print('negatif yok       :', (submission['tuketim'] < 0).sum() == 0)
print()
print(submission['tuketim'].describe().round(1).to_string())

In [ ]:
# Harmanın ay bazında ne yaptığı. Baz çizgisi Mart 2026 ortalaması olduğu için yakın ayı
# yukarı, uzak ayı aşağı çekmesi bekleniyor.
kontrol = submission.copy()
kontrol['ay'] = pd.to_datetime(kontrol['id'].str.rsplit('_', n=1).str[-1]).dt.month
kontrol['sadece_model'] = np.clip(np.expm1(model_log), 0, None)
print(kontrol.groupby('ay')[['sadece_model', 'tuketim']].mean().round(0).to_string())